# 3D-реконструкция объекта по фото — pipeline

**Метод реконструкции:** 3D Gaussian Splatting (Nerfstudio / splatfacto).

## День 1 — минимальный end-to-end pipeline (эта часть ноутбука)

1. Проверка GPU (нужен Colab T4 или лучше)
2. Установка Nerfstudio в изолированное окружение Python 3.11 (через `uv`)
3. Загрузка тестового датасета `dozer` (позы камер уже посчитаны — COLMAP на День 1 не нужен)
4. Обучение splatfacto на всей сцене
5. Экспорт гауссиан → построение меша (Poisson, Open3D) → **STL**

**Критерий успеха Дня 1:** есть STL-файл всей сцены, pipeline работает целиком.

> День 2 (SAM2 + Grounding DINO, маски, реобучение только объекта) добавляется отдельными ячейками ниже после успешного Дня 1.

### Важные замечания по окружению
- Colab на момент работы: **Python 3.13, torch 2.11, CUDA 12.8** — Nerfstudio с этим несовместим напрямую.
- Поэтому Nerfstudio ставится в отдельный venv (`/content/nsenv`) с Python 3.11 и torch 2.1.2 (cu118).
- Ядро ноутбука (3.13) используется только для оркестрации; `ns-train`, `ns-export` и т.д. запускаются из venv как подпроцессы.
- **Перезапуск runtime не нужен** — ядро ноутбука мы не трогаем.
- Выполняйте ячейки строго по порядку.

## Шаг 1. Проверка GPU и окружения

In [ ]:
# Убеждаемся, что Colab выдал GPU. Если тут ошибка или пусто —
# Runtime -> Change runtime type -> Hardware accelerator: GPU (T4).
# ВСЕ строки с ведущим "!" — это shell-команды (не Python).
!nvidia-smi

# Версия Python — пригодится при отладке установки.
import sys
print("Python:", sys.version)

# nvcc может отсутствовать — это нормально, CUDA runtime доступен из torch.
!nvcc --version || echo "nvcc отсутствует (ок для запуска, важно для сборки расширений)"

## Шаг 2. Установка Nerfstudio в изолированном окружении Python 3.11

`pip install nerfstudio` в ядро не работает: Colab на Python 3.13 / torch 2.11, а Nerfstudio нужен
Python ≤3.11 / torch 2.1.x — pip молча откатывается к мусорной версии 0.1.x (2022 г.).

**Стратегия:** через `uv` создаём venv с Python 3.11 → ставим torch 2.1.2 (cu118) → gsplat 1.0.0
(пре-собранное колесо) → Nerfstudio 1.1.5 + open3d + trimesh. Всё в venv `/content/nsenv`.
Команды `ns-*` дальше запускаются как `/content/nsenv/bin/ns-...`.

Ячейки 2a–2e. Установка ~10–15 мин, один раз за сессию. Перезапуск runtime **не нужен**.

In [ ]:
# --- 2a. Диагностика окружения ---
import sys, os
print("Python (ядро ноутбука):", sys.version.split()[0])
try:
    import torch
    print("torch (ядро):", torch.__version__, "| CUDA:", torch.version.cuda)
except ImportError:
    print("torch в ядре не предустановлен")

# Системный CUDA toolkit — нужен, только если gsplat будет собираться из исходников.
# При успешной установке пре-собранного колеса gsplat это не важно.
print("CUDA_HOME:", os.environ.get("CUDA_HOME", "(не задан)"))
!ls -d /usr/local/cuda* 2>/dev/null || echo "нет /usr/local/cuda"
!nvcc --version 2>/dev/null | tail -1 || echo "nvcc не в PATH"

In [ ]:
# --- 2b. Создаём venv с Python 3.11 через uv ---
# uv ставим в ядро ноутбука; он умеет сам скачать standalone-сборку Python 3.11.
!pip install -q uv
!uv python install 3.11
!uv venv --python 3.11 /content/nsenv

# Путь к бинарям venv. ВСЕ последующие ns-команды идут через него.
# Если перезапускали runtime — просто выполните эту ячейку заново (venv на диске сохраняется).
NSPY = "/content/nsenv/bin/python"
NSBIN = "/content/nsenv/bin"
!{NSPY} --version

In [ ]:
# --- 2c. torch 2.1.2 (CUDA 11.8) в venv ---
# Версия, с которой протестирован Nerfstudio 1.1.5. Драйвер Colab новый, cu118 обратно совместим.
# ~2.5 ГБ загрузки, 3-5 мин.
!uv pip install --python /content/nsenv/bin/python "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118

# Проверяем, что torch в venv видит GPU.
!/content/nsenv/bin/python -c "import torch; print('venv torch:', torch.__version__, '| cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))"

In [ ]:
# --- 2d. gsplat 1.0.0 (ядро 3D Gaussian Splatting) в venv ---
# Ставим ПРЕ-СОБРАННОЕ колесо с индекса gsplat для torch 2.1 + cu118 — без компиляции.
# Версия 1.0.0 жёстко закреплена в Nerfstudio 1.1.5.
# --index-url заменяет PyPI, поэтому ставим gsplat ОТДЕЛЬНО, до nerfstudio.
!uv pip install --python /content/nsenv/bin/python "gsplat==1.0.0" --index-url https://docs.gsplat.studio/whl/pt21cu118

# Проверяем импорт (тут же проявится, если колесо несовместимо).
!/content/nsenv/bin/python -c "import gsplat; print('gsplat', gsplat.__version__, 'OK')"

In [ ]:
# --- 2e. Nerfstudio 1.1.5 + Open3D + trimesh в venv ---
# open3d/trimesh ставим в тот же venv (Python 3.11), т.к. в ядре ноутбука Python 3.13
# и для него колёс open3d может не быть.
# gsplat==1.0.0 уже стоит из 2d — pip его не тронет.
# ~8-12 мин: тянет tyro, nerfacc, pymeshlab, viser и т.д.
!uv pip install --python /content/nsenv/bin/python "nerfstudio==1.1.5" open3d trimesh

# Проверяем, что CLI собрался.
!/content/nsenv/bin/ns-train --help > /dev/null 2>&1 && echo "ns-train: OK" || echo "ns-train: ОШИБКА"
print("\n=== Если 'ns-train: OK' — установка Шага 2 завершена. Переходите к Шагу 3. ===")

In [ ]:
# --- 2f. Фикс NumPy ---
# torch 2.1.2 собран под NumPy 1.x; open3d/прочее тянут NumPy 2.x и ломают импорт torch
# ("_ARRAY_API not found"). Жёстко откатываем.
!uv pip install --python /content/nsenv/bin/python "numpy==1.26.4"
!/content/nsenv/bin/python -c "import numpy, torch, gsplat; print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| gsplat', gsplat.__version__)"

## Шаг 3. Проверка venv-окружения

Перезапуск runtime не нужен. Если всё же перезапускали — сначала выполните ячейку **2b** (переопределит `NSPY`/`NSBIN`), затем эту.

In [ ]:
## Шаг 4. Датасет `truck` (Tanks & Temples)

Датасет `dozer` из `ns-download-data` лежит на Google Drive, который блокирует массовые
скачивания с Colab (`gdown FileURLRetrievalError`). Берём вместо него сцену **truck**
из набора авторов 3D Gaussian Splatting — прямая ссылка с `inria.fr`, ~650 МБ,
внутри уже есть готовые COLMAP-позы (`sparse/0/`) и неискажённые кадры (`images/`).

# Скачиваем и распаковываем tandt_db.zip (сцены truck/train/drjohnson/playroom).
!wget -q --show-progress https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip -O /content/tandt_db.zip
!cd /content && unzip -q -o tandt_db.zip

# Структура сцены truck: images/ (полное разрешение), images_2/4/8 (даунскейл), sparse/0/ (COLMAP).
!ls -la /content/tandt/truck
!echo "--- кадров:" && ls /content/tandt/truck/images | wc -l
!echo "--- COLMAP:" && ls -la /content/tandt/truck/sparse/0

In [ ]:
## Шаг 5. Обучение splatfacto (3D Gaussian Splatting)

`max-num-iterations = 7000` вместо дефолтных 30000 — для Дня 1 достаточно, ~15–25 мин на T4. Для финального качества позже поднять до 30000.

Подкоманда `colmap` говорит Nerfstudio читать позы напрямую из `sparse/0`. `--downscale-factor 2` экономит память T4.

## Шаг 5. Обучение splatfacto (3D Gaussian Splatting)

`max-num-iterations = 7000` вместо дефолтных 30000 — для Дня 1 достаточно, ~15–25 мин на T4. Для финального качества позже поднять до 30000.

`--downscale-factor 2` уменьшает изображения — экономит память T4 и ускоряет обучение.

In [ ]:
# Обучение splatfacto. Запуск из venv, в ОДНУ строку.
# yes | ... — авто-ответ на интерактивные вопросы nerfstudio (ячейка не умеет вводить stdin).
# TORCH_CUDA_ARCH_LIST=7.5 / MAX_JOBS=4 — для JIT-компиляции CUDA-ядер gsplat под T4.
# --vis tensorboard — без веб-вьювера (в Colab он недоступен).
# --downscale-factor 1 — кадры truck (~980 px) не даунскейлим; у датасета нет папок images_N,
#   а factor>1 заставил бы nerfstudio интерактивно спрашивать про ресайз.
# Подкоманда colmap с --colmap-path sparse/0 читает позы из COLMAP-модели датасета.
!yes | TORCH_CUDA_ARCH_LIST=7.5 MAX_JOBS=4 /content/nsenv/bin/ns-train splatfacto --data /content/tandt/truck --max-num-iterations 7000 --vis tensorboard --viewer.quit-on-train-completion True --output-dir outputs --pipeline.model.cull-alpha-thresh 0.1 colmap --images-path images --colmap-path sparse/0 --downscale-factor 1

In [ ]:
import glob, os

# Находим самый свежий конфиг обученной модели — путь содержит timestamp запуска.
configs = sorted(glob.glob("outputs/truck/splatfacto/*/config.yml"), key=os.path.getmtime)
assert configs, "config.yml не найден — обучение не завершилось. Смотрите лог выше."
CONFIG = configs[-1]
print("Конфиг обученной модели:", CONFIG)

## Шаг 6. Экспорт гауссиан → меш → STL

`ns-export` для splatfacto отдаёт **не меш**, а набор гауссиан (`.ply` с позициями и цветом). Меш строим отдельно: облако точек → нормали → Screened Poisson reconstruction (Open3D) → STL.

In [ ]:
# Экспорт облака гауссиан в .ply (запуск из venv).
!/content/nsenv/bin/ns-export gaussian-splat --load-config {CONFIG} --output-dir exports/truck
!ls -la exports/truck

In [ ]:
# Скрипт построения меша пишем в файл и запускаем из venv (open3d/trimesh стоят там,
# в ядре ноутбука Python 3.13 — колёс может не быть).
script = r'''
import glob
import numpy as np
import open3d as o3d
import trimesh

# 1. Загружаем экспортированные гауссианы как облако точек.
ply_path = sorted(glob.glob("exports/truck/*.ply"))[-1]
print("PLY:", ply_path)
pcd = o3d.io.read_point_cloud(ply_path)
print("Точек до чистки:", len(pcd.points))

# 2. Убираем плавающие выбросы (артефактные гауссианы вокруг сцены).
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
print("Точек после чистки:", len(pcd.points))

# 3. Poisson-реконструкции нужны нормали — оцениваем по локальным окрестностям
#    и согласуем их направление.
pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)

# 4. Screened Poisson reconstruction -> водонепроницаемый меш.
#    depth=9 — компромисс между детализацией и памятью.
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)

# 5. Обрезаем низкоплотные раздутые края (Poisson «додумывает» поверхность за пределами данных).
densities = np.asarray(densities)
mesh.remove_vertices_by_mask(densities <= np.quantile(densities, 0.05))

# 6. Чистим геометрию и пишем STL (формат требует треугольные нормали).
mesh.remove_degenerate_triangles()
mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles()
mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()

out_stl = "exports/truck/truck_scene.stl"
o3d.io.write_triangle_mesh(out_stl, mesh)
print(f"Сохранено: {out_stl} | вершин: {len(mesh.vertices)} | треугольников: {len(mesh.triangles)}")

# 7. Валидация STL: перечитываем и проверяем базовые метрики качества.
m = trimesh.load(out_stl)
print("watertight:", m.is_watertight, "| euler:", m.euler_number,
      "| объём bbox:", round(float(m.bounding_box.volume), 4),
      "| треугольников:", len(m.faces))
'''

with open("/content/mesh_to_stl.py", "w") as f:
    f.write(script)
print("Скрипт записан: /content/mesh_to_stl.py")

In [ ]:
# Строим меш и STL (запуск из venv).
!cd /content && /content/nsenv/bin/python /content/mesh_to_stl.py

In [ ]:
# Скачать STL. В VS Code Colab extension вместо этого возьмите файл через панель файлов:
# content/exports/truck/truck_scene.stl -> правый клик -> Download
try:
    from google.colab import files
    files.download("exports/truck/truck_scene.stl")
except Exception as e:
    print("Скачивание через API недоступно, возьмите файл из панели файлов:", e)

## Итог Дня 1

✅ **Критерий успеха:** файл `exports/truck/truck_scene.stl` существует, pipeline отработал целиком:
COLMAP-позы → splatfacto (3D Gaussian Splatting) → облако гауссиан → Poisson-меш → STL.

Меш содержит всю сцену (объект + фон + земля) — это ожидаемо. Сегментация объекта — задача Дня 2.

---

## Известные проблемы и обходные пути

**`ns-download-data nerfstudio` падает с `gdown FileURLRetrievalError`**
- Google Drive блокирует эти файлы для Colab. Используем сцену `truck` с `inria.fr` (Шаг 4).

**splatfacto падает на `gsplat.rasterization` (стоит gsplat 1.4.0, nerfstudio писался под 1.0.0)**
- Откат: `uv pip install --python /content/nsenv/bin/python "gsplat==1.0.0"` — версия 1.0.0 есть
  на PyPI как sdist, соберётся из исходников (nvcc 12.8 в системе есть, см. 2a), ~10 мин.

**`ns-train` падает с OOM на T4**
- Увеличьте `--downscale-factor` до 4.
- Добавьте `--pipeline.model.num-downscales 3`.

**Poisson-меш «раздутый» / замкнут в пузырь**
- Поднимите порог обрезки в `mesh_to_stl.py`: `np.quantile(densities, 0.1)` вместо `0.05`.
- Поменяйте `depth` (8 — грубее и быстрее, 10 — детальнее и медленнее).
- Для Дня 2 (только объект) артефактов фона не будет — станет чище.

**`files.download` не работает в VS Code Colab extension**
- Берите файл из панели файлов слева: `content/exports/truck/truck_scene.stl` → правый клик → Download.

**Нужен меш качественнее Poisson**
- Вариант на потом: SuGaR (mesh из Gaussian Splatting) — отдельная установка research-репозитория, в 2-дневный спринт не входит.

## Итог Дня 1

✅ **Критерий успеха:** файл `exports/dozer/dozer_scene.stl` существует, pipeline отработал целиком:
позы камер → splatfacto (3D Gaussian Splatting) → облако гауссиан → Poisson-меш → STL.

Меш содержит всю сцену (объект + фон + пол) — это ожидаемо. Сегментация объекта — задача Дня 2.

---

## Известные проблемы и обходные пути

**`uv pip install gsplat==1.0.0` не находит колесо (2d)**
- Проверьте индекс: попробуйте `--index-url https://docs.gsplat.studio/whl/pt21cu121` (cu121 вместо cu118).
- Крайний вариант — сборка из исходников: убрать `--index-url`, заранее задать `os.environ["CUDA_HOME"]` на путь из ячейки 2a и `os.environ["TORCH_CUDA_ARCH_LIST"]="7.5"`. Долго (~15 мин).

**`nerfstudio==1.1.5` не резолвится в venv (2e)**
- Обычно виноват конфликт с уже стоящими torch/gsplat — проверьте, что 2c и 2d прошли без `error`.
- Можно ослабить: `uv pip install "nerfstudio>=1.1.0,<1.2"`.

**`ns-train` падает с OOM на T4**
- Увеличьте `--downscale-factor` до 4.
- Добавьте `--pipeline.model.num-downscales 3`.

**`ns-export gaussian-splat` дал .ply без нормалей / Poisson-меш «раздутый»**
- Поднимите порог обрезки в `mesh_to_stl.py`: `np.quantile(densities, 0.1)` вместо `0.05`.
- Поменяйте `depth` (8 — грубее и быстрее, 10 — детальнее и медленнее).
- Для Дня 2 (только объект) артефактов фона не будет — станет чище.

**`files.download` не работает в VS Code Colab extension**
- Берите файл из панели файлов слева: `content/exports/dozer/dozer_scene.stl` → правый клик → Download.

**Нужен меш качественнее Poisson**
- Вариант на потом: SuGaR (mesh из Gaussian Splatting) — отдельная установка research-репозитория, в 2-дневный спринт не входит.

In [ ]:
# === ДЕНЬ 2, не запускать в День 1 ===
# COLMAP для оценки поз камер по своим фото (ns-process-data images --data <папка>).
# !sudo apt-get update -qq && sudo apt-get install -y -qq colmap
# !colmap --help | head -n 5

In [ ]:
from PIL import Image
display(Image.open("/content/sparse24_masked_preview.png"))


: 

In [1]:
import os
print("venv:", os.path.exists("/content/nsenv/bin/ns-train"))
print("датасет:", os.path.exists("/content/tandt/truck/images"))
print("маски (Grounded-SAM v1):", os.path.exists("/content/tandt/truck/masks"))
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())


venv: False
датасет: False
маски (Grounded-SAM v1): False
torch: 2.11.0+cu128 | cuda: True
